# Incremental Bronze → Silver Merge

Reads the incremental bronze table and the initial silver table as DataFrames, checks both counts and schemas, verifies schema compatibility, adjusts the incremental bronze dtypes/columns to match silver, then merges (upserts) the incremental records into the silver table and verifies the final count.

In [0]:
%python
# Step 1: Load the incremental bronze records (batch 2 good data) and the current silver table (batch 1 cleaned data)
# into DataFrames, print both row counts, and preview a few incremental rows
bronze_incr_df = spark.read.table('it_support_tickets.bronze.incremental_table')
silver_df = spark.read.table('it_support_tickets.silver.silver_table')

bronze_incr_count = bronze_incr_df.count()
silver_count = silver_df.count()
print('Incremental Bronze count:', bronze_incr_count)
print('Initial Silver count:', silver_count)

display(bronze_incr_df.limit(5))

In [0]:
%python
# Step 2: Print both DataFrames' schemas side by side to visually compare column names
# and data types before attempting any merge
print('=== Incremental Bronze schema ===')
bronze_incr_df.printSchema()
print('=== Initial Silver schema ===')
silver_df.printSchema()

In [0]:
%python
# Step 3: Programmatic schema compatibility check — find (a) shared columns with mismatched dtypes,
# (b) silver columns missing from bronze, and (c) extra bronze columns; the schemas are compatible
# only when all three lists are empty, otherwise Step 4 must fix the differences
bronze_schema = {field.name: field.dataType.simpleString() for field in bronze_incr_df.schema.fields}
silver_schema = {field.name: field.dataType.simpleString() for field in silver_df.schema.fields}

mismatched_dtypes = {
    c: (bronze_schema[c], silver_schema[c])
    for c in bronze_schema
    if c in silver_schema and bronze_schema[c] != silver_schema[c]
}
missing_in_bronze = [c for c in silver_schema if c not in bronze_schema]
extra_in_bronze = [c for c in bronze_schema if c not in silver_schema]

print('Mismatched dtypes:', mismatched_dtypes)
print('Columns missing in incremental bronze:', missing_in_bronze)
print('Extra columns in incremental bronze:', extra_in_bronze)

schema_compatible = (not mismatched_dtypes) and (not missing_in_bronze) and (not extra_in_bronze)
print('Schema compatible:', schema_compatible)

In [0]:
%python
# Step 4: Adjust the incremental bronze DataFrame so its schema matches silver exactly —
# cast mismatched dtypes, derive silver-only columns, and align column order — so the MERGE can run safely
from pyspark.sql.functions import col

adjusted_df = bronze_incr_df

# cast columns whose dtype differs from silver
for field in silver_df.schema.fields:
    name = field.name
    if name in bronze_schema and bronze_schema[name] != field.dataType.simpleString():
        print(f'Casting {name}: {bronze_schema[name]} -> {field.dataType.simpleString()}')
        adjusted_df = adjusted_df.withColumn(name, col(name).cast(field.dataType))

# add silver columns missing from incremental bronze (derived where possible)
if 'total_resolution_cost' in missing_in_bronze:
    adjusted_df = adjusted_df.withColumn(
        'total_resolution_cost', col('resolution_hours') * col('hourly_cost')
    )
    print('Added derived column total_resolution_cost = resolution_hours * hourly_cost')

# align column order to silver
adjusted_df = adjusted_df.select([field.name for field in silver_df.schema.fields])
adjusted_df.printSchema()

In [0]:
%python
# Step 5: Split incremental ticket_ids into new (not in silver yet → MERGE will insert them)
# and matched (already in silver → MERGE will update them); new_count is the expected row growth
new_ids = adjusted_df.select('ticket_id').subtract(silver_df.select('ticket_id'))
matched_ids = adjusted_df.select('ticket_id').intersect(silver_df.select('ticket_id'))

new_count = new_ids.count()
matched_count = matched_ids.count()
print('New ticket_ids to insert:', new_count)
print('Matched ticket_ids to update:', matched_count)

In [0]:
%python
# Step 6: Upsert the adjusted incremental records into silver with a Delta MERGE on ticket_id —
# rows whose ticket_id already exists are fully updated, new ticket_ids are fully inserted
from delta.tables import DeltaTable

silver_dt = DeltaTable.forName(spark, 'it_support_tickets.silver.silver_table')

(silver_dt.alias('target')
    .merge(adjusted_df.alias('source'), 'target.ticket_id = source.ticket_id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

print('MERGE completed into it_support_tickets.silver.silver_table')

In [0]:
%python
# Step 7: Verify the merge — re-count silver and compare against the expected total
# (rows before + new ticket_ids); PASS means existing rows were updated in place rather than duplicated
after_count = spark.read.table('it_support_tickets.silver.silver_table').count()
expected_count = silver_count + new_count

print('Silver count before merge:', silver_count)
print('Silver count after merge:', after_count)
print('Expected count (before + new ticket_ids):', expected_count)

if after_count == expected_count:
    print('PASS: merge count verified')
else:
    print('FAIL: count mismatch, investigate')